# 01. 미 국채 커브 읽기

par yield로 UST 커브를 만들고, 커브 모양과 포지션의 key rate DV01을 읽는다.

- 데이터: `data/market/*.txt` (`날짜 키 값`, `market-data` 레포가 만든다). 없으면 합성 샘플로 돌아가며, 그때 숫자는 해석하지 않는다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

from quanttool.analysis.risk import key_rate
from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes
from quanttool.portfolio import Portfolio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
files = sorted((ROOT / "data" / "market").glob("*.txt"))
SAMPLE = not files
if SAMPLE:
    files = [ROOT / "tests" / "fixtures" / "market_sample.txt"]
    print("경고: 실제 데이터가 없어 합성 샘플을 쓴다. 아래 숫자는 시장 해석용이 아니다.")
cfg = load_config(ROOT / "configs")
quotes = read_quotes(files)
ASOF = list(quotes)[-1]
market = build_market(ASOF, quotes[ASOF], cfg)
ASOF

## 1. 커브 표
- `discount`: 할인계수, `zero_cc`: 연속복리 제로, `fwd_1y`: 그 만기에서 시작하는 1년 선도 (모두 Act/365)
- 표는 이 노트북 안의 작은 함수로 QuantLib 커브에서 바로 읽는다.

In [ ]:
TENORS = ["3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
DC = ql.Actual365Fixed()


def curve_table(curve):
    ref = curve.referenceDate()
    rows = {}
    for t in TENORS:
        d = ref + ql.Period(t)
        rows[t] = {
            "years": DC.yearFraction(ref, d),
            "discount": curve.discount(d),
            "zero_cc": curve.zeroRate(d, DC, ql.Continuous).rate(),
            "fwd_1y": curve.forwardRate(d, d + ql.Period("1Y"), DC, ql.Continuous).rate(),
        }
    return pd.DataFrame(rows).T


def curve_shape(table):
    z = table["zero_cc"] * 1e4
    return pd.Series(
        {
            "level_10y": z["10Y"],
            "2s10s": z["10Y"] - z["2Y"],
            "5s30s": z["30Y"] - z["5Y"],
            "2s5s10s": 2 * z["5Y"] - z["2Y"] - z["10Y"],
        },
        name="zero_bp",
    )


with market.active():
    table = curve_table(market["UST"])
(table[["zero_cc", "fwd_1y"]] * 100).round(3)

In [ ]:
ax = (table[["zero_cc", "fwd_1y"]] * 100).set_index(table["years"]).plot(marker="o")
ax.set_xlabel("years")
ax.set_ylabel("%")
ax.set_title(f"UST curve {ASOF}" + (" (synthetic sample)" if SAMPLE else ""))
plt.show()
curve_shape(table).round(1)

## 2. 모양 지표 추이 (제로 기준, bp)

In [ ]:
history = {}
for d in list(quotes)[-60:]:
    m = build_market(d, quotes[d], cfg)
    with m.active():
        history[d] = curve_shape(curve_table(m["UST"]))
pd.DataFrame(history).T.tail()

## 3. 포지션 key rate DV01

In [ ]:
book = Portfolio.from_file(
    ROOT / "tests" / "fixtures" / "ust_book.yaml", products=cfg.products
).build(market, cfg)
krd = key_rate(book, market, cfg.risk, "UST")["delta"]
print("Σ key rate DV01 =", round(krd["total"].sum(), 2))
ax = krd.drop(columns="total").plot.bar(stacked=True, figsize=(10, 4))
krd["total"].plot(ax=ax, color="k", marker="o", label="total")
ax.set_ylabel("PnL per 1bp (USD)")
ax.legend()
plt.show()

읽는 법: 막대 부호 — 롱은 금리 상승에 손실(음수), 숏은 이익. `total`이 몰린 만기가 실질 듀레이션 위치다. 평행 DV01이 작아도 만기별로 상쇄된 것이면 기울기(2s10s, 5s30s) 위험은 남는다. 이 DV01은 격자 제로금리 기준이다.